# Checking Whether User Input Is a Number or a String

`input()` ALWAYS returns a plain string, no matter what the user typed — even `"10"` comes back as `str`, not `int`. So if you need to know whether the input the user gave you was actually numeric, you have to check that yourself. This notebook covers two approaches: attempting a type conversion (the more robust, generally preferred one), and `str.isdigit()` (simpler, but with real limitations).

**Note:** every `input()` call below is simulated (pre-supplied values, echoed to output) so this notebook runs unattended.

In [ ]:
import builtins

def simulate_input(*responses):

    it = iter(responses)

    def fake_input(prompt=""):

        val = next(it)

        print(f"{prompt}{val}")

        return val

    builtins.input = fake_input



_real_input = input

def restore_input():

    builtins.input = _real_input

## The Problem: `input()` Is Always a String



In [ ]:
simulate_input("10")

number1 = input("Enter number and hit enter ")

print("Printing type of input value")

print("type of number ", type(number1))

restore_input()

## Approach 1: Try Converting It (Recommended)

If a string genuinely represents an integer, `int(s)` succeeds; if it's a decimal number, `float(s)` succeeds; if it's neither, BOTH raise `ValueError` — which you can catch to conclude 'this wasn't a number at all'.

The logic: try `int()` first (catches whole numbers); if that fails, try `float()` (catches decimals); if THAT also fails, it's genuinely just a string.

In [ ]:
def check_user_input(value):

    try:

        val = int(value)

        print("Input is an integer number. Number = ", val)

    except ValueError:

        try:

            val = float(value)

            print("Input is a float  number. Number = ", val)

        except ValueError:

            print("No.. input is not a number. It's a string")



simulate_input("28", "3.14", "28Jessa")



input1 = input("Enter your Age ")

check_user_input(input1)

print()



input2 = input("Enter any number ")

check_user_input(input2)

print()



input3 = input("Enter the last number ")

check_user_input(input3)

restore_input()

- `"28"` -> `int("28")` succeeds immediately -> reported as an integer.
- `"3.14"` -> `int("3.14")` fails (a decimal point isn't valid for `int()`) -> falls through to `float("3.14")`, which succeeds -> reported as a float.
- `"28Jessa"` -> BOTH `int()` and `float()` fail, since letters mixed into a number string are never valid for either -> correctly reported as just a string.

## Approach 2: `str.isdigit()`

`"...".isdigit()` returns `True` only if EVERY character in the string is a digit. It's simpler than the try/except approach, but has a real limitation worth internalizing: **it only works for non-negative WHOLE numbers.** A decimal point, a minus sign, or a plus sign all make it return `False` — so `"3.14".isdigit()` is `False`, and `"-5".isdigit()` is ALSO `False`, even though both are legitimately numbers. This is exactly why the try/except approach above is generally the more reliable, more complete choice.

In [ ]:
def check_is_digit(input_str):

    if input_str.strip().isdigit():

        print("User input is Number")

    else:

        print("User input is string")



simulate_input("45", "45Jessa")



num1 = input("Enter number and hit enter ")

check_is_digit(num1)



num2 = input("Enter number and hit enter ")

check_is_digit(num2)

restore_input()

In [ ]:
# The limitation in action: isdigit() misses negatives AND decimals

print('"3.14".isdigit() ->', "3.14".isdigit())

print('"-5".isdigit()   ->', "-5".isdigit())

print('"5".isdigit()    ->', "5".isdigit())

## Checking an Existing Variable (Not Raw Input) With `isinstance()`

If you already have a Python VALUE (not a string from `input()`) and just want to check whether it's numeric, `isinstance(x, (int, float))` is the direct, idiomatic tool — no string parsing needed at all, since there's no string to parse.

In [ ]:
num = 25.75

print(isinstance(num, (int, float)))



num = '28Jessa'

print(isinstance(num, (int, float)))

## Only Accepting Numeric Input (Keep Asking Until Valid)

Combine the try/except conversion pattern with a `while True` loop to keep re-prompting until the user actually provides something numeric.

In [ ]:
simulate_input("28Jessa", "28")



while True:

    num = input("Please enter a number ")

    try:

        val = int(num)

        print("Input is an integer number.")

        print("Input number is: ", val)

        break

    except ValueError:

        try:

            val = float(num)   # must assign to val here too, or a later reference to val would fail

            print("Input is an float number.")

            print("Input number is: ", val)

            break

        except ValueError:

            print("This is not a number. Please enter a valid number")

restore_input()

**A subtle bug worth flagging:** in the inner `float(num)` branch, the conversion result MUST be assigned to `val` (`val = float(num)`), not just called as `float(num)` on its own — a version that calls `float(num)` without assigning it would later try to print a `val` that was never actually set on that code path, raising `NameError`. This is a good reminder that `try` succeeding doesn't automatically mean whatever you computed got stored anywhere.

## Practice Problem: Positive or Negative?

Combine the numeric-conversion check with a simple sign comparison.

In [ ]:
simulate_input("42")

user_number = input("Enter your number ")



print()

try:

    val = int(user_number)

    if val > 0:

        print("User number is positive")

    else:

        print("User number is negative")

except ValueError:

    print("No.. input string is not a number. It's a string")

restore_input()

## Summary

- `input()` always returns a `str` — you must explicitly attempt conversion to know if it's really a number.
- The try/except `int()`-then-`float()` pattern is the most robust way to classify a string as int, float, or 'not a number' — it correctly handles negatives, decimals, and mixed alphanumeric strings.
- `str.isdigit()` is simpler but ONLY correctly identifies non-negative whole numbers — it wrongly reports `False` for both negative numbers and decimals.
- For an already-existing Python value (not a raw input string), `isinstance(x, (int, float))` is the right, direct tool.
- Wrapping the conversion attempt in `while True` gives you a 'keep asking until valid' input loop.

## Practice Exercises

1. Write a function `classify(s)` that returns the string `"int"`, `"float"`, or `"not a number"` for a given input string (don't just print — actually return the classification).
2. Test `str.isdigit()` against these five strings and predict the result before running: `"007"`, `"3.0"`, `"-3"`, `""`, `"  42  "` (note the surrounding spaces).
3. Write a function `sum_valid_numbers(strings)` that takes a list of strings, and returns the sum of only the ones that successfully convert to a number (int or float), skipping the rest.
4. Simulate a loop that keeps asking for a number between 1 and 10 (inclusive), re-prompting on both non-numeric input AND out-of-range numeric input, until a valid one is given.
5. Using `isinstance()`, write a function `describe_type(x)` that reports whether `x` is `bool`, `int`, `float`, or 'other' — being careful that `True`/`False` are technically a SUBCLASS of `int` in Python, so check for `bool` FIRST.

### Exercise 1 solution

In [ ]:
def classify(s):

    try:

        int(s)

        return "int"

    except ValueError:

        try:

            float(s)

            return "float"

        except ValueError:

            return "not a number"



for s in ["42", "3.14", "hello", "-7"]:

    print(s, "->", classify(s))

### Exercise 2 solution

In [ ]:
tests = ["007", "3.0", "-3", "", "  42  "]

for t in tests:

    print(f"{t!r}.isdigit() -> {t.isdigit()}")

### Exercise 3 solution

In [ ]:
def sum_valid_numbers(strings):

    total = 0

    for s in strings:

        try:

            total += int(s)

        except ValueError:

            try:

                total += float(s)

            except ValueError:

                print(f"Skipping non-numeric value: {s!r}")

    return total



print(sum_valid_numbers(["10", "abc", "2.5", "xyz", "7"]))

### Exercise 4 solution

In [ ]:
simulate_input("abc", "15", "7")



while True:

    raw = input("Enter a number between 1 and 10: ")

    try:

        val = int(raw)

    except ValueError:

        print("Not a valid number, try again.")

        continue

    if 1 <= val <= 10:

        print("Got a valid number:", val)

        break

    else:

        print("Out of range, try again.")

restore_input()

### Exercise 5 solution

In [ ]:
def describe_type(x):

    if isinstance(x, bool):        # must check bool BEFORE int - bool is a subclass of int!

        return "bool"

    elif isinstance(x, int):

        return "int"

    elif isinstance(x, float):

        return "float"

    else:

        return "other"



for value in [True, 42, 3.14, "hello", [1, 2]]:

    print(value, "->", describe_type(value))